In [4]:
import os
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

In [6]:
image_dir = Path("./miniddsm/Benign")


In [7]:
image_dir

PosixPath('miniddsm/Benign')

In [10]:
# 1) Show where Python is currently looking from
print("Current working dir:", Path.cwd())
print("Resolved image_dir:", image_dir.resolve())

Current working dir: /Users/helmutremse/tempMini/23_final_3070/code/explore
Resolved image_dir: /Users/helmutremse/tempMini/23_final_3070/code/explore/miniddsm/Benign


In [11]:
# 2) Check that the directory exists and is accessible
print("Exists:", image_dir.exists())
print("Is directory:", image_dir.is_dir())

Exists: True
Is directory: True


In [14]:
# 3) Try to find image files
exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".dcm"}
files = [p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in exts]

print("Image-like files found:", len(files))
print("First 5 files:")
for p in files[:5]:
    print(" -", p.name)
    

Image-like files found: 4
First 5 files:
 - C_0087_1.LEFT_MLO_Mask3.png
 - B_3460_1.RIGHT_CC_Mask3.png
 - C_0078_1.LEFT_CC_Mask5.png
 - B_3460_1.RIGHT_MLO_Mask3.png


In [16]:
import pandas as pd
from pathlib import Path

# Supported image extensions
exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".dcm"}

# Find the dataset root whether notebook is run from project root or explore/
candidates = [Path("miniddsm"), Path("../miniddsm")]
dataset_root = next((p for p in candidates if p.exists() and p.is_dir()), None)

if dataset_root is None:
    raise FileNotFoundError(
        "Could not find 'miniddsm' folder. Tried: "
        + ", ".join(str(p.resolve()) for p in candidates)
    )

rows = []
for label in ["Benign", "Cancer"]:
    class_dir = dataset_root / label
    if not class_dir.exists():
        continue

    for p in class_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            rows.append(
                {
                    "image_path": p.as_posix(),
                    "label": label,
                }
            )

# Build DataFrame and save CSV
csv_df = pd.DataFrame(rows).sort_values("image_path").reset_index(drop=True)
out_csv = Path("miniddsm_image_labels.csv")
csv_df.to_csv(out_csv, index=False)

print(f"Dataset root: {dataset_root.resolve()}")
print(f"Rows written: {len(csv_df)}")
print(f"CSV saved to: {out_csv.resolve()}")
csv_df.head()

Dataset root: /Users/helmutremse/tempMini/23_final_3070/code/explore/miniddsm
Rows written: 8
CSV saved to: /Users/helmutremse/tempMini/23_final_3070/code/explore/miniddsm_image_labels.csv


,image_path,label
0,miniddsm/Benign/B_3460_1.RIGHT_CC_Mask3.png,Benign
1,miniddsm/Benign/B_3460_1.RIGHT_MLO_Mask3.png,Benign
2,miniddsm/Benign/C_0078_1.LEFT_CC_Mask5.png,Benign
3,miniddsm/Benign/C_0087_1.LEFT_MLO_Mask3.png,Benign
4,miniddsm/Cancer/B_3034_1.LEFT_MLO_Mask3.png,Cancer


In [17]:
import pandas as pd

df = pd.read_csv("miniddsm_image_labels.csv")
print(df.head())
print(df.columns)
print("Number of rows:", len(df))

                                     image_path   label
0   miniddsm/Benign/B_3460_1.RIGHT_CC_Mask3.png  Benign
1  miniddsm/Benign/B_3460_1.RIGHT_MLO_Mask3.png  Benign
2    miniddsm/Benign/C_0078_1.LEFT_CC_Mask5.png  Benign
3   miniddsm/Benign/C_0087_1.LEFT_MLO_Mask3.png  Benign
4   miniddsm/Cancer/B_3034_1.LEFT_MLO_Mask3.png  Cancer
Index(['image_path', 'label'], dtype='object')
Number of rows: 8


In [18]:
df.info

<bound method DataFrame.info of                                      image_path   label
0   miniddsm/Benign/B_3460_1.RIGHT_CC_Mask3.png  Benign
1  miniddsm/Benign/B_3460_1.RIGHT_MLO_Mask3.png  Benign
2    miniddsm/Benign/C_0078_1.LEFT_CC_Mask5.png  Benign
3   miniddsm/Benign/C_0087_1.LEFT_MLO_Mask3.png  Benign
4   miniddsm/Cancer/B_3034_1.LEFT_MLO_Mask3.png  Cancer
5    miniddsm/Cancer/B_3045_1.LEFT_CC_Mask3.png  Cancer
6  miniddsm/Cancer/C_0022_1.RIGHT_MLO_Mask3.png  Cancer
7   miniddsm/Cancer/C_0025_1.RIGHT_CC_Mask6.png  Cancer>

In [5]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

# Load the CSV we created earlier
csv_df = pd.read_csv("miniddsm_image_labels.csv")

# Map string labels to integers for PyTorch
label_to_idx = {"Benign": 0, "Cancer": 1}

class MiniDDSMImageDataset(Dataset):
    def __init__(self, dataframe, label_map, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.label_map = label_map
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = Path(row["image_path"])
        # to get a consistent image
        image = Image.open(image_path).convert("RGB")
        label = self.label_map[row["label"]]

        if self.transform is not None:
            image = self.transform(image)

        return image, label

# Create the dataset object, but not the DataLoader yet
dataset = MiniDDSMImageDataset(csv_df, label_to_idx)

print("Dataset size:", len(dataset))
image, label = dataset[0]
print("First sample label:", label)
print("First sample image size:", image.size)
print(label_to_idx["Benign"])

Dataset size: 8
First sample label: 0
First sample image size: (1548, 2276)
0


In [6]:
# Define a simple preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Recreate the dataset with the transform attached
dataset = MiniDDSMImageDataset(csv_df, label_to_idx, transform=transform)

# Check one sample
image, label = dataset[0]
print("Transformed image shape:", image.shape)
print("Transformed image dtype:", image.dtype)
print("Label:", label)

Transformed image shape: torch.Size([3, 224, 224])
Transformed image dtype: torch.float32
Label: 0


In [7]:
from torch.utils.data import DataLoader

# Wrap the dataset in a DataLoader and inspect one batch
loader = DataLoader(dataset, batch_size=4, shuffle=True)
images, labels = next(iter(loader))
print("Batch image shape:", images.shape)
print("Batch labels:", labels)
print("Batch size:", len(labels))

Batch image shape: torch.Size([4, 3, 224, 224])
Batch labels: tensor([0, 0, 1, 0])
Batch size: 4
